# Final Demo - One-Year Real Dataset Pipeline

This notebook is the final large-data demonstration path. PR1 and PR2 remain progress-review notebooks; this notebook shows how the same production jobs scale to a full year across all US airports available in the ARCO-ERA5/BTS lakehouse.

The notebook is safe by default: it inventories state and prints the one-year command. Set `RUN_PIPELINE=True` only when intentionally running the full 12-month build.


In [1]:
YEAR = 2024
RUN_PIPELINE = False
WITH_DELTA = False
WITH_FINAL_MODEL = False
CALL_REAL_GOLD_ROW_API = True
API_URL = 'http://aviation-api:3000/predict'
print('Year:', YEAR)
print('Run full one-year pipeline now:', RUN_PIPELINE)


Year: 2024
Run full one-year pipeline now: False


## Raw and Lakehouse Inventory

This cell checks whether raw data and lakehouse outputs exist for each month. It uses MinIO object listings, so it is lightweight.


In [2]:
import os
import boto3
import pandas as pd

endpoint = os.environ['MINIO_ENDPOINT_INTERNAL']
s3 = boto3.client(
    's3',
    endpoint_url=endpoint,
    aws_access_key_id=os.environ['MINIO_ROOT_USER'],
    aws_secret_access_key=os.environ['MINIO_ROOT_PASSWORD'],
    region_name=os.environ.get('AWS_REGION', 'us-east-1'),
)

def object_count(bucket, prefix):
    total = 0
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        total += len(page.get('Contents', []))
    return total

rows = []
for month in range(1, 13):
    rows.append({
        'month': f'{month:02d}',
        'raw_weather_objects': object_count('raw', f'arco_era5_us_airport_hourly/year={YEAR}/month={month:02d}/'),
        'raw_bts_zip_objects': object_count('raw', f'bts_on_time/raw_zip/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{YEAR}_{month}.zip'),
        'bronze_weather_objects': object_count('lakehouse', f'bronze/weather/year={YEAR}/month={month:02d}/'),
        'bronze_bts_objects': object_count('lakehouse', f'bronze/bts_on_time/year={YEAR}/month={month:02d}/'),
        'silver_objects': object_count('lakehouse', f'silver/flight_weather_daily/year={YEAR}/month={month:02d}/'),
        'gold_objects': object_count('lakehouse', f'gold/training_features/year={YEAR}/month={month:02d}/'),
        'delta_gold_objects': object_count('lakehouse', f'gold_delta/training_features/year={YEAR}/month={month:02d}/'),
    })
inventory = pd.DataFrame(rows)
display(inventory)
print('Months with raw weather:', int((inventory.raw_weather_objects > 0).sum()))
print('Months with raw BTS ZIP:', int((inventory.raw_bts_zip_objects > 0).sum()))
print('Months with Gold features:', int((inventory.gold_objects > 0).sum()))


,month,raw_weather_objects,raw_bts_zip_objects,bronze_weather_objects,bronze_bts_objects,silver_objects,gold_objects,delta_gold_objects
0,01,7,1,6,3,4,3,5
1,02,7,1,6,3,4,3,5
2,03,7,1,6,3,4,3,5
3,04,7,1,6,3,4,3,5
4,05,7,1,6,4,4,3,5
5,06,7,1,6,4,4,3,5
6,07,7,1,6,4,4,3,5
7,08,7,1,6,4,4,3,5
8,09,7,1,6,3,4,3,5
9,10,7,1,6,4,4,3,5


Months with raw weather: 12
Months with raw BTS ZIP: 12
Months with Gold features: 12


## Full-Year Runner

This is the command for the full one-year pipeline. It reuses the same production Spark modules as PR2 for every month and can optionally build Delta and register a full-year model.


In [3]:
import subprocess, sys

command = [
    sys.executable, '-m', 'spark_jobs.run_year_pipeline',
    '--year', str(YEAR),
]
if WITH_DELTA:
    command.append('--with-delta')
if WITH_FINAL_MODEL:
    command.append('--with-final-model')

print('Full-year command:')
print(' '.join(command))
if RUN_PIPELINE:
    subprocess.run(command, check=True, cwd='/workspace')
else:
    print('Safe presentation mode: command shown but not executed.')


Full-year command:
/opt/conda/bin/python -m spark_jobs.run_year_pipeline --year 2024
Safe presentation mode: command shown but not executed.


## API Call From a Real Gold Feature Row

This cell satisfies the final-demo requirement to call the deployed API with a real data point from the lakehouse feature table.


In [4]:
import subprocess, sys

if CALL_REAL_GOLD_ROW_API:
    subprocess.run([
        sys.executable, '-m', 'spark_jobs.call_api_with_gold_sample',
        '--year', str(YEAR),
        '--month', '1',
        '--api-url', API_URL,
    ], check=True, cwd='/workspace')
else:
    print('Skipped real Gold-row API call.')


## Optional Live Weather Inference

For an extra live demo, run this from the VM while the API is up:

```bash
docker compose exec jupyter bash -lc 'cd /workspace && python -m api.live_weather_predict --airport JFK --api-url http://aviation-api:3000/predict'
```

This fetches current Open-Meteo weather for the airport coordinates in the metadata table, transforms it to the model feature contract, and calls the same BentoML endpoint.
